In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schemas, supported seasons, and football-data.co.uk league
# mappings are loaded from 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.match_results_football_data"

SOURCE_SYSTEM = "football-data.co.uk"

# One Silver row represents one football match.
#
# match_date is included in the business key to make match identity explicit
# and protect against any future case where the same home/away pairing appears
# more than once within a league season.
BUSINESS_KEY = [
    "season_code",
    "league_name",
    "match_date",
    "home_team",
    "away_team",
]

REQUIRED_SILVER_COLUMNS = [
    "season_code",
    "season_start_year",
    "league_name",
    "match_date",
    "home_team",
    "away_team",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# football-data.co.uk Bronze maintains one table per league.
# Each table contains all supported seasons for that league.

LEAGUES = list(FOOTBALL_DATA_LEAGUES.values())

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.football_data_{league_name}"
    for league_name in LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# Unlike the FBref Silver notebooks, missing columns are intentionally allowed
# during the union.
#
# football-data.co.uk changes its CSV schema over time, particularly betting
# and bookmaker columns. Bronze preserves that source evolution, while this
# Silver transformation consumes only a stable analytical subset.
#
# Missing league TABLES are still treated as errors.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
    allow_missing_tables=False,
    allow_missing_columns=True,
)


# -----------------------------------------------------------------------------
# Validate expected source columns
# -----------------------------------------------------------------------------
# Although the overall football-data.co.uk schema varies over time, the fields
# below are required by this Silver transformation.
#
# If one of these disappears, the pipeline should fail explicitly rather than
# silently produce an incomplete analytical table.

REQUIRED_BRONZE_COLUMNS = [
    "Div",
    "Date",
    "Time",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "Referee",
    "HS",
    "AS",
    "HST",
    "AST",
    "HC",
    "AC",
    "HF",
    "AF",
    "HY",
    "AY",
    "HR",
    "AR",
    "B365H",
    "B365D",
    "B365A",
    "_season",
    "_league",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# football-data.co.uk Bronze deliberately stores source business columns as
# strings because inferSchema=False is used during Bronze ingestion.
#
# Trim whitespace and convert empty strings to NULL before casting fields into
# their Silver datatypes.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Select, rename, clean, and cast Silver columns
# -----------------------------------------------------------------------------
# Silver converts the source-oriented CSV schema into clearer analytical names
# and explicit datatypes.
#
# Match dates are converted from source strings into a proper Spark DATE.
# Both common two-digit and four-digit year representations are supported.

silver_df = (
    bronze_df
    .select(
        # ---------------------------------------------------------------------
        # Match identity
        # ---------------------------------------------------------------------

        clean_string("Div")
        .alias("division_code"),

        F.coalesce(
            F.expr("try_to_date(Date, 'dd/MM/yyyy')"),
            F.expr("try_to_date(Date, 'dd/MM/yy')"),
        ).alias("match_date"),

        clean_string("Time")
        .alias("kickoff_time"),

        clean_string("HomeTeam")
        .alias("home_team"),

        clean_string("AwayTeam")
        .alias("away_team"),

        # ---------------------------------------------------------------------
        # Full-time result
        # ---------------------------------------------------------------------

        F.col("FTHG")
        .cast("int")
        .alias("full_time_home_goals"),

        F.col("FTAG")
        .cast("int")
        .alias("full_time_away_goals"),

        clean_string("FTR")
        .alias("full_time_result"),


        # ---------------------------------------------------------------------
        # Half-time result
        # ---------------------------------------------------------------------

        F.col("HTHG")
        .cast("int")
        .alias("half_time_home_goals"),

        F.col("HTAG")
        .cast("int")
        .alias("half_time_away_goals"),

        clean_string("HTR")
        .alias("half_time_result"),


        # ---------------------------------------------------------------------
        # Match officials
        # ---------------------------------------------------------------------

        clean_string("Referee")
        .alias("referee_name"),


        # ---------------------------------------------------------------------
        # Shots
        # ---------------------------------------------------------------------

        F.col("HS")
        .cast("int")
        .alias("home_shots"),

        F.col("AS")
        .cast("int")
        .alias("away_shots"),

        F.col("HST")
        .cast("int")
        .alias("home_shots_on_target"),

        F.col("AST")
        .cast("int")
        .alias("away_shots_on_target"),


        # ---------------------------------------------------------------------
        # Corners
        # ---------------------------------------------------------------------

        F.col("HC")
        .cast("int")
        .alias("home_corners"),

        F.col("AC")
        .cast("int")
        .alias("away_corners"),


        # ---------------------------------------------------------------------
        # Fouls
        # ---------------------------------------------------------------------

        F.col("HF")
        .cast("int")
        .alias("home_fouls"),

        F.col("AF")
        .cast("int")
        .alias("away_fouls"),


        # ---------------------------------------------------------------------
        # Cards
        # ---------------------------------------------------------------------

        F.col("HY")
        .cast("int")
        .alias("home_yellow_cards"),

        F.col("AY")
        .cast("int")
        .alias("away_yellow_cards"),

        F.col("HR")
        .cast("int")
        .alias("home_red_cards"),

        F.col("AR")
        .cast("int")
        .alias("away_red_cards"),


        # ---------------------------------------------------------------------
        # Bet365 pre-match odds
        # ---------------------------------------------------------------------
        # Only the selected Bet365 1X2 odds are promoted into Silver.
        #
        # The many additional bookmaker columns remain available in Bronze and
        # can be introduced later if there is a concrete analytical use case.

        F.col("B365H")
        .cast("double")
        .alias("bet365_home_win_odds"),

        F.col("B365D")
        .cast("double")
        .alias("bet365_draw_odds"),

        F.col("B365A")
        .cast("double")
        .alias("bet365_away_win_odds"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------
        # _league already contains the canonical project league identifier
        # assigned during ingestion, for example:
        #
        #   premier_league
        #   bundesliga
        #   serie_a

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        clean_string("_league")
        .alias("league_name"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future entity standardization placeholders
# -----------------------------------------------------------------------------
# The next project phase will standardize entities across:
#
#   football-data.co.uk
#   FBref
#   API-Football
#
# Important candidates from this dataset include:
#
#   home_team
#   away_team
#   referee_name
#   league_name
#
# Team-name standardization will be particularly important because equivalent
# clubs may use different source representations, for example abbreviations,
# accented names, or different naming conventions.
#
# Preferred future model:
#
#   home_team_name_source
#   home_team_name_standardized
#   home_team_key
#
#   away_team_name_source
#   away_team_name_standardized
#   away_team_key
#
#   competition_key
#
# Referee standardization can be considered separately if it becomes useful
# analytically.
#
# No physical placeholder columns are added yet because empty or duplicated
# fields would add unnecessary schema noise before matching logic exists.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Every match must have:
#
#   season
#   league
#   date
#   home team
#   away team
#
# Validation occurs before deduplication so records with incomplete match keys
# cannot be grouped together and reduced arbitrarily.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning a Bronze load against the
# same CSV files can create repeated versions of the same match.
#
# Keep the version from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# Required analytical identifiers are validated as the final data-quality
# gate.
#
# The helper also returns the final row count, avoiding an additional count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"match records into {SILVER_TABLE}."
)